# Modul 06 · nb03 — Fairness, Bias & Tata Kelola

> **Domain sertifikasi (NCA-GENL): Trustworthy AI (10%) — bias & fairness, alignment & transparency.**
> Ini awal **Track 2** modul ini. Track 1 (nb01–nb02) mengajarkan *bagaimana* membuat LLM cepat dan men-*deploy*-nya. Track 2 menanyakan hal yang jauh lebih penting: **boleh tidak kita lepas sistem ini ke publik?** Cepat saja tidak cukup — sistem yang cepat tapi **bias** atau **tak bisa dipertanggungjawabkan** justru lebih berbahaya, karena ia salah dengan skala dan kecepatan penuh.

## Peta perjalanan Track 2

Trustworthy AI bukan satu fitur yang bisa ditempel di akhir. Ia rangkaian lapisan, dan modul ini membahasnya berurutan:

| Notebook | Pilar | Kapan bekerja |
|---|---|---|
| **nb03 (ini)** | **Fairness · Transparency · Accountability** | **Offline** — audit *sebelum* dan *sesudah* deploy |
| nb04 | Safety · Privacy (guardrails NemoGuard) | **Runtime** — memeriksa tiap pesan |
| nb05 | Capstone — semua lapisan dirangkai jadi `/ask` berpagar | Produksi |

Perhatikan kata **offline** di baris nb03. Fairness dan tata kelola **tidak** dikerjakan saat pengguna menekan tombol kirim. Keduanya dikerjakan di meja audit — dengan data, angka, dan dokumen — *sebelum* model dipercaya, lalu diulang berkala. Itu sebabnya nb03 berdiri sendiri, terpisah dari rail runtime di nb04.

Di notebook ini kamu akan:
1. mengenal **lima pilar Trustworthy AI** versi NVIDIA dan memetakan modul ke pilar-pilar itu;
2. **menghitung fairness sungguhan** (demographic parity, equalized odds) dengan numpy/pandas pada dataset kecil bikinan sendiri — bukan teori, tapi angka yang bisa kamu reproduksi;
3. mendeteksi bias pada **teks bebas** dengan **LLM-judge** lewat panggilan NIM nyata (Nemotron-3-nano), lalu mengontraskan **metrik deterministik vs judge yang fleksibel**;
4. menulis dua artefak tata kelola yang nyata: **Model Card** dan **AI Ethics Checklist**, dengan kaitan **UU PDP No. 27/2022**.

## 0. Setup — install + kunci API

Hanya satu hal "berat" di notebook ini: panggilan **LLM-judge** ke NVIDIA NIM. Sisanya murni numpy/pandas dan menulis file — ringan, jalan di CPU, tanpa GPU.

Kuncinya **jangan ditulis di kode**. Ambil dari **Colab Secrets** (ikon 🔑 di panel kiri), nama persis `NVIDIA_API_KEY`. Gratis tanpa kartu kredit di [build.nvidia.com](https://build.nvidia.com): *sign in → buka model mana saja → **Get API Key***. Key kamu diawali `nvapi-`.

In [ ]:
# Install klien OpenAI (yang berbicara ke NIM). numpy/pandas/sklearn sudah ada di Colab.
!pip install -q "openai>=1.40"

import os, getpass

# Ambil NVIDIA_API_KEY dari Colab Secrets (JANGAN hardcode key di sel).
try:
    from google.colab import userdata
    os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY") or ""
except Exception:
    if not os.environ.get("NVIDIA_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA_API_KEY (nvapi-...): ")

assert os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"), \
    "Key harus diawali 'nvapi-'. Tambahkan di Colab Secrets (ikon kunci kiri) dulu."
print("NVIDIA_API_KEY siap.")

## 1. Lima pilar Trustworthy AI (NVIDIA)

NVIDIA merangkum "AI yang bisa dipercaya" dalam beberapa pilar yang berulang di dokumentasi NeMo Guardrails dan materi Trustworthy AI mereka. Versi ringkasnya — lima pilar yang menjadi tulang punggung Track 2:

| Pilar | Pertanyaan inti | Di mana ditangani di modul ini |
|---|---|---|
| **Nondiscrimination / Fairness** | Apakah model **adil** lintas kelompok (gender, suku, wilayah)? | nb03 §2 — fairness metrics |
| **Transparency** | Bisakah orang luar **memahami** apa model ini, datanya, dan batasannya? | nb03 §4 — Model Card |
| **Accountability** | **Siapa** bertanggung jawab kalau sistem salah? Ada jejak audit? | nb03 §4 — ethics checklist |
| **Safety** | Apakah sistem menolak menghasilkan konten berbahaya? | nb04 — content-safety rail |
| **Privacy & Security** | Apakah data pribadi (PII) terlindungi? Ada *consent*? | nb04 — PII masking + UU PDP |

Dua hal penting tentang pembagian ini:

**Pertama, pilar-pilar ini dikerjakan di waktu yang berbeda.** Safety dan Privacy adalah penjaga **runtime** — mereka memeriksa setiap pesan saat sistem berjalan (itu pekerjaan nb04). Tapi Fairness, Transparency, dan Accountability adalah pekerjaan **audit offline**: kamu mengukurnya di meja kerja, dengan dataset historis, *sebelum* sistem dipercaya dan secara berkala *setelah* deploy. Tidak ada "rail fairness" yang menyala tiap permintaan — fairness diuji pada ribuan keputusan sekaligus, bukan satu per satu.

**Kedua, fairness tidak terlihat dari akurasi.** Sebuah model bisa punya akurasi total 92% dan tetap **diam-diam tidak adil**: salah hampir selalu pada satu kelompok. Akurasi total menyembunyikan itu; fairness metrics-lah yang membongkarnya. Inilah yang akan kita buktikan dengan angka di bagian berikut.

## 2. Fairness metrics — dihitung NYATA

### Skenario: model penilai pinjaman

Bayangkan sebuah bank memakai model untuk memutuskan **siapa yang disetujui pinjamannya**. Setiap pemohon punya:
- `gender` — kelompok yang ingin kita periksa keadilannya (di Indonesia ini relevan untuk pengawasan **OJK** dan UU anti-diskriminasi);
- `layak` — label *kebenaran*: apakah pemohon ini **sebenarnya** sanggup membayar (1) atau tidak (0);
- `setuju` — **keputusan model**: disetujui (1) atau ditolak (0).

Kita sengaja menyuntikkan bias: **perempuan yang sebenarnya layak lebih sering ditolak** daripada laki-laki yang sama-sama layak. Ini meniru bias dunia nyata yang muncul dari data historis yang timpang. Tugas kita: **menangkap bias itu dengan angka**, bukan dengan firasat.

### Dua metrik klasik

1. **Demographic parity** — apakah **tingkat persetujuan** sama antar kelompok? Bandingkan P(setuju | Pria) vs P(setuju | Wanita). *Disparitas* = selisih maksimum antar kelompok.

2. **Equalized odds** — lebih ketat dan lebih adil: apakah model sama akuratnya di tiap kelompok? Bandingkan dua angka antar kelompok:
   - **TPR** (*True Positive Rate*) — dari yang **benar-benar layak**, berapa proporsi yang disetujui? (peluang yang pantas tidak terlewat)
   - **FPR** (*False Positive Rate*) — dari yang **tidak layak**, berapa proporsi yang keliru disetujui?

> **Ambang ajar: disparitas < 0.1 dianggap cukup adil.** Angka ini bukan hukum alam — ia ambang praktis yang umum dipakai untuk *menandai* sistem yang perlu ditinjau. Lewat 0.1 = lampu kuning, audit lebih dalam.

Mengapa equalized odds "lebih adil" dari demographic parity? Demographic parity bisa **menipu**: ia bisa terpenuhi dengan menyetujui orang-orang *tidak layak* dari kelompok yang dirugikan hanya untuk menyamakan angka — itu adil di permukaan tapi merusak. Equalized odds menuntut keadilan **bersyarat pada kebenaran**: di antara orang yang sama-sama layak, peluang disetujui harus sama. Itu sebabnya kita laporkan keduanya.

In [ ]:
# Dataset kecil BIKINAN: 1.000 pengajuan pinjaman, dengan bias gender disuntikkan sengaja.
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(0)              # seed tetap -> hasil bisa direproduksi
n = 1000
gender = rng.choice(["Pria", "Wanita"], size=n)
layak  = rng.integers(0, 2, size=n)         # 1 = SEBENARNYA layak (label kebenaran)

# Model "ideal" = setuju persis kalau layak. Lalu kita RUSAK keadilannya:
setuju = layak.copy()
bias_mask = (gender == "Wanita") & (layak == 1)          # perempuan yang sebenarnya layak
setuju[bias_mask] = rng.choice([0, 1], size=bias_mask.sum(), p=[0.40, 0.60])
# -> 40% perempuan layak DITOLAK walau pantas. Laki-laki layak tidak diutak-atik.

df = pd.DataFrame({"gender": gender, "layak": layak, "setuju": setuju})
print(df.head(8).to_string(index=False))
print(f"\nTotal pemohon: {n}  |  perempuan-layak yang sengaja dirugikan: {int(bias_mask.sum())}")
print(f"Akurasi total model (setuju==layak): {(df.setuju == df.layak).mean():.1%}  <- terlihat 'bagus', kan?")

Perhatikan baris terakhir: akurasi total model terlihat tinggi. **Itu jebakannya.** Sekarang kita bongkar dengan fairness metrics — kita tulis kedua fungsi dari nol supaya jelas tidak ada sihir, hanya hitungan proporsi.

In [ ]:
def demographic_parity(y_pred, group):
    """Tingkat persetujuan per kelompok + disparitas (selisih maksimum antar kelompok)."""
    rates = {str(g): round(float(y_pred[group == g].mean()), 3) for g in np.unique(group)}
    disparity = round(max(rates.values()) - min(rates.values()), 3)
    return rates, disparity

def equalized_odds(y_true, y_pred, group):
    """TPR & FPR per kelompok + disparitas masing-masing."""
    per_group = {}
    for g in np.unique(group):
        m = group == g
        tn, fp, fn, tp = confusion_matrix(y_true[m], y_pred[m], labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) else 0.0       # dari yang layak, berapa disetujui
        fpr = fp / (fp + tn) if (fp + tn) else 0.0       # dari yang tak layak, berapa keliru disetujui
        per_group[str(g)] = {"TPR": round(tpr, 3), "FPR": round(fpr, 3)}
    tpr_gap = round(max(v["TPR"] for v in per_group.values()) - min(v["TPR"] for v in per_group.values()), 3)
    fpr_gap = round(max(v["FPR"] for v in per_group.values()) - min(v["FPR"] for v in per_group.values()), 3)
    return per_group, tpr_gap, fpr_gap

THRESHOLD = 0.10   # ambang ajar: disparitas di atas ini = perlu ditinjau

rates, dp_gap = demographic_parity(df.setuju.values, df.gender.values)
print("== Demographic parity ==")
print("  tingkat persetujuan per kelompok:", rates)
print(f"  disparitas = {dp_gap}   -> {'LULUS (< 0.1)' if dp_gap < THRESHOLD else 'GAGAL: TIDAK ADIL (>= 0.1)'}")

eo, tpr_gap, fpr_gap = equalized_odds(df.layak.values, df.setuju.values, df.gender.values)
print("\n== Equalized odds ==")
for grp, v in eo.items():
    print(f"  {grp:7s} TPR={v['TPR']}  FPR={v['FPR']}")
print(f"  gap TPR = {tpr_gap}  -> {'LULUS' if tpr_gap < THRESHOLD else 'GAGAL: peluang disetujui TIDAK SAMA untuk yang sama-sama layak'}")
print(f"  gap FPR = {fpr_gap}  -> {'LULUS' if fpr_gap < THRESHOLD else 'perlu ditinjau'}")

### Membaca hasilnya dengan jujur

Hasilnya **TIDAK ADIL**, dan angkanya menunjukkan persis di mana:

- **Demographic parity** — tingkat persetujuan perempuan jatuh di bawah laki-laki, disparitas tembus 0.1. Lampu merah.
- **Equalized odds** — yang paling tajam adalah **gap TPR**: di antara pemohon yang *sama-sama benar-benar layak*, perempuan punya peluang disetujui yang lebih kecil. Inilah definisi diskriminasi yang sebenarnya — bukan "perempuan lebih jarang disetujui" (bisa saja memang lebih sedikit yang layak), melainkan "perempuan **layak** lebih sering ditolak daripada laki-laki **layak**".

Bandingkan ini dengan **akurasi total yang terlihat tinggi (sekitar 88%)** yang kita cetak tadi. Akurasi menyembunyikan bias karena ia menjumlahkan semua kelompok jadi satu angka. Fairness metrics memecahnya per kelompok — di situlah bias tertangkap.

> **Konteks Indonesia.** Untuk *credit scoring* (diawasi **OJK**) dan rekrutmen, keputusan otomatis yang bias bisa melanggar aturan anti-diskriminasi. UU PDP No. 27/2022 juga memberi individu **hak keberatan atas keputusan yang dibuat sepenuhnya secara otomatis** — artinya sistem seperti ini wajib bisa diaudit dan dijelaskan, bukan kotak hitam. Fairness metrics adalah bukti audit itu.

## 3. Deteksi bias pada teks bebas — LLM-judge (NIM nyata)

Fairness metrics di §2 bekerja indah **untuk keputusan terstruktur**: ada label, ada prediksi, ada kelompok — semuanya angka. Tapi bagaimana dengan **teks bebas**? Misalnya keluaran sebuah chatbot, deskripsi lowongan kerja, atau materi pelatihan yang ingin kita saring dari stereotip. Tidak ada `confusion_matrix` untuk kalimat.

Di sinilah pendekatan kedua masuk: **LLM sebagai juri (LLM-judge)**. Kita minta sebuah model menilai apakah sepotong teks mengandung bias/stereotip. Ide ini sama dengan metrik bias berbasis LLM di tool seperti DeepEval — bedanya, kita arahkan ke **NIM** kita sendiri (Nemotron-3-nano), tanpa OpenAI berbayar.

### Mekanik NVIDIA yang WAJIB ditampilkan: mematikan reasoning

Nemotron-3-nano adalah **reasoning model** (arsitektur MoE Hybrid Mamba-Attention, 30B total / ~3B aktif). Kalau dibiarkan, ia menjawab dengan blok `<think>...</think>` panjang sebelum jawaban final — boros token, dan mengotori output yang ingin kita parse jadi sekadar "Yes"/"No".

Di endpoint NIM, trik prompt `/no_think` **DIABAIKAN**. Satu-satunya cara mematikan reasoning adalah lewat parameter request:

```python
extra_body={"chat_template_kwargs": {"enable_thinking": False}}
```

Ini mekanik paling NVIDIA-specific di seluruh modul, jadi kita **tampilkan utuh di bawah** — bukan disembunyikan di balik helper. (Notebook nb04 dan nb05 akan memanggil pola yang sama; di sana ia boleh dibungkus helper karena kamu sudah melihat aslinya di sini.)

In [ ]:
from openai import OpenAI

# SATU klien OpenAI yang berbicara ke NVIDIA NIM. base_url + key; nanti cukup ganti model=.
client = OpenAI(base_url="https://integrate.api.nvidia.com/v1",
                api_key=os.environ["NVIDIA_API_KEY"])

JUDGE_MODEL = "nvidia/nemotron-3-nano-30b-a3b"   # reasoning model -> WAJIB matikan thinking

def nemotron(messages, max_tokens=64, temperature=0.0):
    """Panggil Nemotron via NIM. extra_body mematikan reasoning -> output bersih & cepat.
    DITAMPILKAN UTUH (tidak disembunyikan) karena inilah mekanik paling NVIDIA-specific di modul ini."""
    r = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},  # token /no_think DIABAIKAN NIM
    )
    return (r.choices[0].message.content or "").strip()

# Smoke test: dengan flag, jawaban langsung final (tanpa <think>...).
print(nemotron([{"role": "user", "content": "Jawab satu kata: apa lawan kata 'adil'?"}], max_tokens=16))

Sekarang kita bungkus `nemotron()` menjadi **bias judge**: beri ia peran penilai, sodorkan teks, dan paksa jawaban menjadi `Yes`/`No` saja agar mudah dipakai program.

In [ ]:
def bias_judge(teks: str) -> bool:
    """True jika teks mengandung stereotip/bias kelompok. Memakai Nemotron via NIM."""
    jawab = nemotron(
        max_tokens=8,
        messages=[
            {"role": "system",
             "content": "Kamu penilai bias yang teliti. Tugasmu menilai apakah sebuah teks "
                        "mengandung stereotip atau bias terhadap kelompok (gender, suku, agama, "
                        "ras, daerah, usia). Generalisasi merendahkan terhadap kelompok = bias."},
            {"role": "user",
             "content": f'Teks: "{teks}"\nApakah teks ini mengandung bias/stereotip kelompok? '
                        f'Jawab HANYA satu kata: Yes atau No.'},
        ],
    )
    return jawab.strip().lower().startswith("yes")

contoh = [
    "Programmer yang hebat biasanya laki-laki.",                       # bias gender
    "Programmer yang hebat menguasai algoritma dan rajin berlatih.",   # netral (kriteria, bukan kelompok)
    "Orang dari daerah itu memang malas bekerja.",                     # bias daerah
    "Kandidat ini punya 5 tahun pengalaman backend dan kuat di SQL.",  # netral
    "Perempuan kurang cocok jadi pemimpin tim teknis.",               # bias gender
]
print("vonis   | teks")
print("--------+" + "-" * 60)
for t in contoh:
    label = "BIAS" if bias_judge(t) else "netral"
    print(f"{label:7s} | {t}")

### Metrik deterministik vs judge yang fleksibel — kontras inti

Kita baru saja memakai **dua alat yang berbeda jenis** untuk satu tujuan (mendeteksi ketidakadilan). Memahami kapan memakai yang mana adalah inti pelajaran ini:

| | **Fairness metrics (§2)** | **LLM-judge (§3)** |
|---|---|---|
| **Input** | keputusan terstruktur (label, prediksi, kelompok) | teks bebas |
| **Sifat** | **deterministik** — input sama → angka sama, selalu | **probabilistik** — bisa beda antar panggilan |
| **Bisa salah?** | tidak (asal rumus benar) | **ya** — judge juga model, bisa keliru/halu |
| **Bisa dijelaskan?** | penuh — tiap angka punya rumus | sebagian — "menurut model" |
| **Cocok untuk** | credit scoring, rekrutmen, klasifikasi | moderasi teks, audit konten, kasus tak terstruktur |
| **Biaya** | nol (numpy) | panggilan API per teks |

**Pelajaran:** jangan memilih satu dan membuang yang lain. Untuk **keputusan berisiko tinggi yang terstruktur** (siapa dapat pinjaman), pakai **metrik deterministik** — kamu butuh angka yang bisa dipertanggungjawabkan di hadapan auditor OJK, bukan "kata si LLM". Untuk **teks bebas dalam skala besar** (menyaring jutaan komentar), LLM-judge adalah satu-satunya yang praktis — tapi perlakukan keluarannya sebagai **sinyal**, bukan vonis, dan validasi dengan sampel manusia.

> ⚠️ **Judge bisa keliru.** Karena `bias_judge` adalah LLM, jawabannya bisa berubah antar panggilan dan bisa salah pada kalimat ambigu. Itu sebabnya kita pakai `temperature=0` (sekonsisten mungkin) dan tetap menyebutnya **sinyal**. Untuk audit resmi, gabungkan dengan tinjauan manusia.

## 4. Tata kelola: Transparency & Accountability

Dua pilar terakhir nb03 — **Transparency** dan **Accountability** — tidak diukur dengan angka. Keduanya diwujudkan sebagai **dokumen (deliverable)** yang nyata dan sering dinilai di *review* kelayakan rilis. Tanpa dokumen ini, sistem AI adalah kotak hitam yang tak bisa dipertanggungjawabkan — persis yang dilarang regulasi modern.

Dua artefak standar industri:

1. **Model Card** — "label gizi" sebuah model: apa modelnya, data apa yang dipakai, untuk apa ia dimaksudkan, dan **apa batasannya**. Konsep ini dipopulerkan Google dan kini standar de facto (Hugging Face menampilkannya di tiap repo model). → memenuhi **Transparency**.

2. **AI Ethics Checklist** — daftar periksa yang ditandatangani sebelum rilis: privasi, keamanan, fairness, dan **siapa penanggung jawabnya**. → memenuhi **Accountability**.

Kita tulis keduanya sebagai file nyata di sel berikut — bukan teks tempel, melainkan *deliverable* yang bisa kamu serahkan ke tim audit.

In [ ]:
# (1) Model Card — "label gizi" model. Kita tulis untuk bot RAG yang akan kita deploy di nb05.
import json

model_card = {
    "nama": "navasena-rag-bot",
    "versi": "0.1",
    "base_model": "nvidia/nemotron-3-nano-30b-a3b (via NVIDIA NIM)",
    "tanggal": "2026-06",
    "penanggung_jawab": "Tim ML Navasena  <ml@contoh.id>",   # Accountability: ADA nama, bukan anonim
    "intended_use": "Asisten tanya-jawab dokumen internal berbahasa Indonesia (RAG).",
    "out_of_scope": [
        "BUKAN nasihat hukum / medis / finansial",
        "BUKAN untuk keputusan otomatis berisiko tinggi tanpa pengawasan manusia",
    ],
    "data": "Hanya dokumen yang di-index pengguna; tidak ada PII pelatihan tambahan.",
    "guardrails": [
        "input: self-check jailbreak + content-safety (nb04)",
        "input: topic-control (nb04)",
        "output: content-safety + grounding (nb04)",
        "PII masking Indonesia: NIK/HP/rekening (nb04)",
    ],
    "fairness": "Diaudit demographic parity & equalized odds (ambang disparitas < 0.1, lihat §2).",
    "batasan_diketahui": [
        "Judge/LLM bisa keliru -> jawaban harus diverifikasi dengan sumber (sitasi RAG).",
        "Cakupan jawaban terbatas pada dokumen yang di-index.",
    ],
    "privasi": "Mengikuti UU PDP No. 27/2022; PII di-mask sebelum disimpan/masuk log.",
}

with open("MODEL_CARD.json", "w") as f:
    json.dump(model_card, f, indent=2, ensure_ascii=False)
print(json.dumps(model_card, indent=2, ensure_ascii=False))
print("\nTersimpan: MODEL_CARD.json")

In [ ]:
# (2) AI Ethics Checklist — deliverable Accountability. Baris privasi mengikat ke UU PDP.
checklist = """# AI Ethics Checklist — navasena-rag-bot

## 1. Privasi (UU PDP No. 27/2022)
- [ ] Consent diperoleh sebelum memproses data pribadi
- [ ] PII di-mask sebelum disimpan / masuk log (NIK, no. HP, rekening)
- [ ] Tersedia mekanisme HAK KEBERATAN atas keputusan otomatis (Pasal hak subjek data)
- [ ] Rencana notifikasi kebocoran data dalam <= 72 jam ke pihak berwenang & subjek data

## 2. Keamanan & Keselamatan (Safety)
- [ ] Input rail: deteksi jailbreak / prompt-injection (nb04)
- [ ] Input rail: content-safety (taksonomi Aegis, nb04)
- [ ] Output rail: blokir konten berbahaya sebelum sampai ke pengguna (nb04)

## 3. Transparansi (Transparency)
- [ ] Model Card tersedia & mutakhir (model, data, batasan)
- [ ] Jawaban menyertakan sitasi sumber (RAG) agar bisa diverifikasi

## 4. Non-diskriminasi (Fairness)
- [ ] Uji demographic parity (disparitas < 0.1)
- [ ] Uji equalized odds — gap TPR & FPR < 0.1 antar kelompok
- [ ] Dataset audit cukup representatif lintas kelompok

## 5. Akuntabilitas (Accountability)
- [ ] Penanggung jawab sistem tercantum jelas (nama + kontak)
- [ ] Log & jejak audit aktif dan tersimpan
- [ ] Prosedur eskalasi bila sistem salah sudah disepakati
"""

with open("AI_ETHICS_CHECKLIST.md", "w") as f:
    f.write(checklist)
print(checklist)
print(f"Tersimpan: AI_ETHICS_CHECKLIST.md ({len(checklist)} karakter)")

### Kaitan UU PDP No. 27/2022 (Indonesia)

Tiga baris di checklist bukan formalitas — ketiganya **mengikat secara hukum** untuk sistem yang memproses data warga Indonesia:

- **Notifikasi kebocoran ≤ 72 jam.** Bila terjadi kegagalan perlindungan data pribadi, pengendali wajib memberitahukan ke pihak berwenang (dan subjek data) paling lambat **72 jam**. Itu sebabnya log dan jejak audit (pilar Accountability) bukan opsional — tanpa log, kamu tak tahu *apa* yang bocor dalam 72 jam itu.
- **Denda hingga 2% pendapatan tahunan.** Pelanggaran administratif dapat dikenai denda hingga **2% dari pendapatan tahunan**. Biaya membangun guardrails (nb04) jauh lebih kecil dari ini.
- **Hak keberatan atas keputusan otomatis.** Subjek data berhak keberatan terhadap keputusan yang dibuat *sepenuhnya* otomatis yang menimbulkan akibat hukum/signifikan. Inilah jembatan langsung antara §2 (fairness) dan hukum: model kredit yang bias dan tak bisa diaudit melanggar prinsip ini.

> Tiga pilar yang kita kerjakan di nb03 — Fairness, Transparency, Accountability — bukan "nilai tambah". Untuk konteks Indonesia, ketiganya adalah **prasyarat kepatuhan**.

## Yang kita pelajari

- **Lima pilar Trustworthy AI (NVIDIA)** — Fairness, Transparency, Accountability, Safety, Privacy. nb03 mengerjakan tiga yang pertama (audit **offline**); nb04 mengerjakan dua sisanya (rail **runtime**).
- **Fairness diukur, bukan ditebak.** Kita menghitung **demographic parity** dan **equalized odds** dengan numpy/pandas pada dataset bikinan, dan melihat bias tertangkap (disparitas ≥ 0.1) padahal **akurasi total terlihat tinggi**. Akurasi menyembunyikan bias; fairness metrics membongkarnya per kelompok.
- **Equalized odds lebih ketat** dari demographic parity karena bersyarat pada kebenaran (gap TPR pada yang sama-sama layak) — lebih sulit ditipu.
- **Dua jenis deteksi, dua konteks.** Metrik **deterministik** untuk keputusan terstruktur (bisa dipertanggungjawabkan, nol biaya); **LLM-judge** (NIM, Nemotron-3-nano) untuk teks bebas (fleksibel tapi bisa keliru — perlakukan sebagai sinyal).
- **Mekanik NIM yang nyata** — kita menampilkan `extra_body={"chat_template_kwargs": {"enable_thinking": False}}` **utuh**, bukan disembunyikan. Inilah cara mematikan reasoning Nemotron (token `/no_think` diabaikan NIM).
- **Tata kelola = deliverable.** **Model Card** (Transparency) dan **AI Ethics Checklist** (Accountability) adalah file nyata, dikaitkan ke **UU PDP No. 27/2022** (notifikasi ≤ 72 jam, denda ≤ 2%, hak keberatan).

### Berikutnya: nb04 — Safety, Guardrails & Privasi (NemoGuard NYATA)

nb03 mengaudit sistem **di meja kerja**. nb04 memasang **penjaga runtime** yang memeriksa **setiap pesan**: kita jalankan **NemoGuard NIM sungguhan** (content-safety dengan taksonomi Aegis, topic-control), self-check jailbreak via NeMo Guardrails OSS, lalu PII masking + grounding. Dari audit offline → ke pagar yang menyala saat pengguna mengetik.

## 🧪 Latihan

Kerjakan langsung di notebook ini — semua bahan sudah tersedia.

1. **Geser tingkat bias.** Pada sel dataset, ganti `p=[0.40, 0.60]` menjadi `p=[0.10, 0.90]` (perempuan layak lebih jarang ditolak), lalu jalankan ulang §2. Pada nilai berapa disparitas demographic parity turun di bawah 0.1 dan sistem dinyatakan **LULUS**? Coba juga `p=[0.50, 0.50]` — apa yang terjadi pada gap TPR?

2. **Tambah kelompok ketiga.** Tambahkan kolom `wilayah` (`"Kota"`/`"Desa"`) ke `df`, suntikkan bias pada salah satunya, lalu panggil `demographic_parity` dan `equalized_odds` dengan `group=df.wilayah.values`. Fairness bisa diuji untuk **atribut apa pun** yang sensitif — buktikan fungsimu sudah generik.

3. **Uji ketahanan judge.** Beri `bias_judge` lima kalimat **ambigu** buatanmu (mis. "Anak muda zaman sekarang lebih melek teknologi."). Jalankan tiap kalimat **3 kali**. Apakah vonisnya selalu sama? Diskusikan: mengapa ini membuktikan judge adalah **sinyal**, bukan vonis — dan mengapa untuk keputusan kredit kita tetap memilih metrik deterministik di §2.

4. **Lengkapi tata kelola.** Tambahkan satu pilar **lingkungan** ke `AI_ETHICS_CHECKLIST.md` (mis. "estimasi jejak karbon inference dilaporkan") dan satu field `"limitations_environmental"` ke `MODEL_CARD.json`. Trustworthy AI yang utuh juga mempertimbangkan dampak lingkungan komputasi.